<a href="https://colab.research.google.com/github/lakshya701/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lakshya701/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*
**The rule, in plain words:** A page is flagged as a refresh candidate when it is
*stale* (hasn't been updated in 180+ days) AND still *visible* (getting 500+ impressions
in the last 90 days). Score = impressions_90d when both conditions hold, else 0 — so
among flagged pages, the ones pulling more traffic while stale rank higher.

**Reason codes this rule can output:**
- `stale_visible_page` — meets both conditions, flagged for review
- `none` — doesn't meet the threshold, not flagged (goes to `monitor`)


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/lakshya701/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

print("Current directory:", os.getcwd())

import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

df["baseline_action_score"] = (
    (df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)
).astype(int) * df["impressions_90d"]

df["reason_code"] = "stale_visible_page"
df.loc[df["baseline_action_score"] == 0, "reason_code"] = "none"

df["action"] = "refresh_candidate"
df.loc[df["baseline_action_score"] == 0, "action"] = "monitor"

queue = df[df["baseline_action_score"] > 0].sort_values("baseline_action_score", ascending=False)

os.makedirs("work/outputs", exist_ok=True)
output_cols = ["content_id", "baseline_action_score", "reason_code", "action",
               "impressions_90d", "days_since_last_update", "avg_position", "ctr"]
queue[output_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)

print(f"{len(queue):,} pages flagged out of {len(df):,} total")
queue[output_cols].head(20)

Current directory: /content/flyrank-ml-internship/flyrank-ml-internship
17 pages flagged out of 30,000 total


,content_id,baseline_action_score,reason_code,action,impressions_90d,days_since_last_update,avg_position,ctr
16751,content_cf56e2e2e282,61678,stale_visible_page,refresh_candidate,61678,194,19.7,0.15
16514,content_7368877ea310,59472,stale_visible_page,refresh_candidate,59472,194,24.8,0.13
7021,content_1bfaa38ff26c,25715,stale_visible_page,refresh_candidate,25715,194,22.2,0.23
21268,content_0a91db491d14,13299,stale_visible_page,refresh_candidate,13299,193,10.5,0.49
11489,content_5feee3994adb,7812,stale_visible_page,refresh_candidate,7812,194,39.0,0.01
12045,content_c2d929d83eaa,7558,stale_visible_page,refresh_candidate,7558,193,17.9,0.20
698,content_b16bd7307b39,4590,stale_visible_page,refresh_candidate,4590,194,31.0,0.00
5327,content_fe16a55cd13d,4556,stale_visible_page,refresh_candidate,4556,194,16.4,0.33
26810,content_ecb6215e79fd,4429,stale_visible_page,refresh_candidate,4429,194,25.3,0.38
20837,content_928af3e22c80,1697,stale_visible_page,refresh_candidate,1697,193,15.8,0.12


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*
1. content_cf56e2e2e282 | refresh_candidate | stale_visible_page | score=61,678 |
   High confidence — very high impressions despite being stale. Wrong if: this page
   recently had a URL/ownership change and the "last update" timestamp is a data
   artifact rather than real content neglect.
2. content_7368877ea310 | refresh_candidate | stale_visible_page | score=59,472 |
   High confidence, same reasoning. Wrong if: content type is evergreen (e.g. a
   reference page) that doesn't need frequent updates to stay relevant.
3. content_1bfaa38ff26c | refresh_candidate | stale_visible_page | score=25,715 |
   High confidence. Wrong if: a sibling page absorbed its traffic (consolidation),
   not a genuine decline.
4. content_0a91db491d14 | refresh_candidate | stale_visible_page | score=13,299 |
   Medium-high confidence. Wrong if: seasonal demand naturally dipped and will
   recover without intervention.
5. content_5feee3994adb | refresh_candidate | stale_visible_page | score=7,812 |
   Medium confidence — score drops off noticeably here. Wrong if: position is
   already strong and staleness isn't actually hurting visibility yet.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*
**Which picks look wrong and why:** With only 17 pages flagged out of 30,000, this rule
is extremely conservative — it likely misses many genuinely declining pages that don't
happen to be both stale (180+ days) AND still high-visibility (500+ impressions). It
also treats all stale pages identically regardless of how stale.

**Leakage check:** Confirmed no product flags (health_score, priority_score, action_type)
were used — none exist in this dataset. Confirmed no future-window data was used — every
input is a trailing, already-observed measurement, not derived from a future outcome.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.